# DATATHON 2026 — The Gridbreakers
## Phần 3: Mô hình Dự báo Doanh thu (Sales Forecasting)

**Pipeline:** Feature Engineering → LightGBM + TimeSeriesSplit CV → Ensemble → SHAP Explainability

| | |
|---|---|
| Model | LightGBM (log1p target transform) |
| Validation | TimeSeriesSplit (5 folds, time-aware — không data leakage) |
| Metrics | MAE · RMSE · R² |
| Explainability | SHAP TreeExplainer (Summary Plot + Bar Chart) |
| Reproducibility | `SEED = 42` được đặt nhất quán cho numpy, LightGBM, SHAP |

---
## 0. Imports & Seed toàn cục

> **Reproducibility note:** `SEED = 42` được truyền vào tất cả thành phần có tính ngẫu nhiên (numpy, LightGBM). Đặt ở đây một lần duy nhất để dễ kiểm soát.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
%matplotlib inline

# ── Global seed ──────────────────────────────────────────────────────────────
# Đặt một lần duy nhất, truyền vào mọi thành phần có tính ngẫu nhiên
SEED = 42
np.random.seed(SEED)

print(f'SEED = {SEED}  ✅')

---
## 1. Cấu hình đường dẫn

In [ ]:
DATA_DIR   = Path('.')          # ← chỉnh lại nếu chạy ở môi trường khác
TRAIN_FILE = DATA_DIR / 'sales.csv'
TEST_FILE  = DATA_DIR / 'sample_submission.csv'
OUT_DIR    = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

TARGET_REV  = 'Revenue'
TARGET_COGS = 'COGS'

print(f'Train : {TRAIN_FILE}')
print(f'Test  : {TEST_FILE}')
print(f'Output: {OUT_DIR.resolve()}')

---
## 2. Load dữ liệu

In [ ]:
sales      = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
sample_sub = pd.read_csv(TEST_FILE,  parse_dates=['Date'])

print(f'Train : {sales.shape}  |  {sales.Date.min().date()} → {sales.Date.max().date()}')
print(f'Test  : {sample_sub.shape}  |  {sample_sub.Date.min().date()} → {sample_sub.Date.max().date()}')
print()
display(sales[['Revenue', 'COGS']].describe().round(2))

---
## 3. Feature Engineering

Tất cả đặc trưng được tạo **chỉ từ cột `Date`** — không sử dụng Revenue/COGS từ tập test, đảm bảo không có data leakage.

| Nhóm | Features | Mục đích |
|---|---|---|
| Calendar | year, month, day, dayofweek, dayofyear, quarter | Nắm bắt chu kỳ thời gian cơ bản |
| Flags | is_weekend, is_payday | Hành vi mua sắm đặc thù |
| Fourier (năm) | sin/cos_year_1,2,3 | Mùa vụ trong năm (Tết, sale mùa hè…) |
| Fourier (tuần) | sin/cos_week | Mùa vụ trong tuần |
| Fourier (tháng) | month_sin, month_cos | Chu kỳ tháng |


In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tạo toàn bộ đặc trưng từ cột Date.
    Hàm thuần túy — không side-effect, dễ kiểm tra và tái lập.
    """
    df = df.copy()
    d  = df['Date']

    # ── Calendar ─────────────────────────────────────────────────────────────
    df['year']      = d.dt.year
    df['month']     = d.dt.month
    df['day']       = d.dt.day
    df['dayofweek'] = d.dt.dayofweek   # 0=Thứ Hai … 6=Chủ Nhật
    df['dayofyear'] = d.dt.dayofyear
    df['quarter']   = d.dt.quarter

    # ── Flags hành vi ────────────────────────────────────────────────────────
    df['is_weekend'] = d.dt.dayofweek.isin([5, 6]).astype(int)
    # Ngày đầu/cuối tháng thường trùng ngày lương → doanh thu cao hơn
    df['is_payday']  = d.dt.day.isin([1, 2, 3, 28, 29, 30, 31]).astype(int)

    # ── Fourier: mùa vụ trong năm ────────────────────────────────────────────
    # k=1,2,3 để nắm bắt nhiều harmonic (Tết Q1, sale hè Q2, 11/11 Q4…)
    doy = d.dt.dayofyear
    for k in [1, 2, 3]:
        df[f'sin_year_{k}'] = np.sin(2 * np.pi * k * doy / 365.25)
        df[f'cos_year_{k}'] = np.cos(2 * np.pi * k * doy / 365.25)

    # ── Fourier: mùa vụ trong tuần ───────────────────────────────────────────
    dow = d.dt.dayofweek
    df['sin_week'] = np.sin(2 * np.pi * dow / 7)
    df['cos_week'] = np.cos(2 * np.pi * dow / 7)

    # ── Fourier: tháng trong năm ─────────────────────────────────────────────
    df['month_sin'] = np.sin(2 * np.pi * d.dt.month / 12)
    df['month_cos'] = np.cos(2 * np.pi * d.dt.month / 12)

    return df


df_train = build_features(sales)
df_test  = build_features(sample_sub)

EXCLUDES = ['Date', 'Revenue', 'COGS']
FEATURES = [c for c in df_train.columns if c not in EXCLUDES]

print(f'Số features: {len(FEATURES)}')
print(f'Features   : {FEATURES}')

---
## 4. Hàm tiện ích

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Tính MAE, RMSE, R² trên không gian giá trị gốc."""
    return {
        'MAE' : mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2'  : r2_score(y_true, y_pred),
    }


def print_cv_table(fold_metrics: list, target_name: str) -> None:
    """In bảng kết quả CV theo từng fold + Mean ± Std."""
    W = 62
    print(f"\n{'═'*W}")
    print(f"  KẾT QUẢ CV — {target_name.upper()}")
    print(f"{'═'*W}")
    print(f"  {'Fold':>4}  │  {'MAE':>13}  │  {'RMSE':>13}  │  {'R²':>7}")
    print(f"  {'─'*55}")
    for i, m in enumerate(fold_metrics, 1):
        print(f"  {i:>4}  │  {m['MAE']:>13,.0f}  │  {m['RMSE']:>13,.0f}  │  {m['R2']:>7.4f}")
    maes  = [m['MAE']  for m in fold_metrics]
    rmses = [m['RMSE'] for m in fold_metrics]
    r2s   = [m['R2']   for m in fold_metrics]
    print(f"  {'─'*55}")
    print(f"  {'Mean':>4}  │  {np.mean(maes):>13,.0f}  │  {np.mean(rmses):>13,.0f}  │  {np.mean(r2s):>7.4f}")
    print(f"  {'Std':>4}  │  {np.std(maes):>13,.0f}  │  {np.std(rmses):>13,.0f}  │  {np.std(r2s):>7.4f}")
    print(f"{'═'*W}\n")


print('✅ Utility functions sẵn sàng')

---
## 5. Training — LightGBM + TimeSeriesSplit CV

**Thiết kế tránh data leakage:**
- `TimeSeriesSplit` đảm bảo fold validation luôn nằm **sau** fold train về mặt thời gian.
- Target được biến đổi `log1p` trước khi train (phân phối Revenue lệch phải mạnh); inverse lại bằng `expm1` khi đánh giá.
- `random_state=SEED` được truyền vào LightGBM để đảm bảo tái lập.

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# Ghi rõ tất cả params để dễ tái lập và review
LGB_PARAMS = dict(
    n_estimators     = 1500,
    learning_rate    = 0.03,
    max_depth        = 7,
    num_leaves       = 31,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = SEED,   # ← seed nhất quán
    importance_type  = 'gain',
    verbosity        = -1,
)

N_SPLITS       = 5
EARLY_STOPPING = 100

tscv = TimeSeriesSplit(n_splits=N_SPLITS)


def train_lgb(target_name: str) -> list:
    """
    Train LightGBM với 5-fold TimeSeriesSplit.
    Trả về list models (1 model/fold) để dùng cho ensemble.
    """
    models       = []
    fold_metrics = []
    X = df_train[FEATURES]

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X), 1):
        X_tr = X.iloc[train_idx]
        X_va = X.iloc[val_idx]

        # log1p transform để ổn định phân phối lệch phải
        y_tr     = np.log1p(df_train[target_name].iloc[train_idx])
        y_va_log = np.log1p(df_train[target_name].iloc[val_idx])
        y_va_act = df_train[target_name].iloc[val_idx].values

        model = lgb.LGBMRegressor(**LGB_PARAMS)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va_log)],
            callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False)],
        )

        preds = np.expm1(model.predict(X_va))   # inverse log1p
        m     = compute_metrics(y_va_act, preds)
        fold_metrics.append(m)
        models.append(model)

        print(f'  Fold {fold}/{N_SPLITS} — '
              f'MAE: {m["MAE"]:>12,.0f}  '
              f'RMSE: {m["RMSE"]:>12,.0f}  '
              f'R²: {m["R2"]:.4f}')

    print_cv_table(fold_metrics, target_name)
    return models


print('▶ Training Revenue ...')
models_rev  = train_lgb(TARGET_REV)

print('▶ Training COGS ...')
models_cogs = train_lgb(TARGET_COGS)

---
## 6. SHAP — Giải thích mô hình bằng ngôn ngữ kinh doanh

SHAP (SHapley Additive exPlanations) cho phép đo lường **đóng góp thực sự** của từng feature vào từng dự báo, thay vì chỉ xem feature importance trung bình.

### Cách đọc biểu đồ:
- **Bar chart (trái):** Top features theo mức độ ảnh hưởng trung bình đến dự báo Revenue.
- **Beeswarm (phải):** Mỗi chấm = một ngày. Màu đỏ = giá trị feature cao, xanh = thấp. Vị trí trục X = SHAP value (dương → đẩy Revenue lên, âm → kéo xuống).

### Diễn giải kinh doanh (sẽ được phân tích sau khi chạy):
- **`year`** — Xu hướng tăng trưởng dài hạn của doanh nghiệp.
- **`sin_year_1`, `cos_year_1`** — Mùa vụ trong năm: đỉnh Q4 (11/11, Giáng Sinh) và Tết Q1.
- **`dayofweek`, `is_weekend`** — Hành vi mua sắm cuối tuần vs ngày thường.
- **`is_payday`** — Người dùng chi tiêu nhiều hơn xung quanh ngày nhận lương.
- **`month`** — Ảnh hưởng theo tháng cụ thể (tháng 6, tháng 12 thường cao).

In [ ]:
def plot_shap_explainability(models: list, target_name: str, sample_n: int = 2000) -> None:
    """
    Vẽ SHAP bar chart + beeswarm plot cho model fold cuối.
    Dùng sample ngẫu nhiên cố định (random_state=SEED) để tái lập.
    """
    # Dùng model fold cuối — đã thấy nhiều dữ liệu nhất
    model    = models[-1]
    X_sample = df_train[FEATURES].sample(
        min(sample_n, len(df_train)), random_state=SEED  # ← seed cố định
    )

    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

    fig, ax = plt.subplots(figsize=(10, 7))
    plt.sca(ax)

    # ── Bar chart: mean |SHAP| ────────────────────────────────────────────────
    shap.summary_plot(shap_values, X_sample,
                      plot_type='bar', show=False, max_display=15)
    ax.set_title(
        f'Yếu tố dẫn động {target_name} — mean |SHAP value|\n'
        f'(SHAP TreeExplainer, sample n={sample_n}, seed={SEED})',
        fontsize=12, fontweight='bold'
    )
    ax.set_xlabel('mean(|SHAP value|) — mức độ ảnh hưởng trung bình đến dự báo')
    plt.tight_layout()

    save_path = OUT_DIR / f'shap_{target_name.lower()}.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  💾 Đã lưu → {save_path}')


print('--- SHAP: Revenue ---')
plot_shap_explainability(models_rev,  TARGET_REV)

print('--- SHAP: COGS ---')
plot_shap_explainability(models_cogs, TARGET_COGS)

### Diễn giải SHAP theo ngôn ngữ kinh doanh

Sau khi xem biểu đồ, có thể điền vào bảng dưới đây cho báo cáo:

| Feature | Ý nghĩa kinh doanh | Tác động |
|---|---|---|
| `year` | Xu hướng tăng trưởng dài hạn — doanh thu tăng theo năm | Dương, mạnh |
| `sin_year_1` / `cos_year_1` | Mùa vụ trong năm: đỉnh Q4 (11/11, Noel) và Tết Q1 | Rất cao |
| `month` | Các tháng cuối năm (11, 12) và Tết (1, 2) có doanh thu cao hơn | Cao |
| `dayofweek` / `is_weekend` | Cuối tuần: doanh thu thường thấp hơn ngày thường | Trung bình |
| `is_payday` | Người dùng mua sắm nhiều hơn vào đầu/cuối tháng (ngày lương) | Dương |
| `quarter` | Q4 luôn là quý đỉnh — cần tồn kho và logistics sẵn sàng | Cao |

---
## 7. Dự báo tập Test & Ensemble

In [ ]:
def ensemble_predict(models: list, X_test: pd.DataFrame) -> np.ndarray:
    """
    Trung bình dự báo của 5 fold models.
    Giảm variance so với single-model prediction.
    """
    preds = np.array([np.expm1(m.predict(X_test)) for m in models])
    return preds.mean(axis=0)


X_test = df_test[FEATURES]

pred_rev  = ensemble_predict(models_rev,  X_test)
pred_cogs = ensemble_predict(models_cogs, X_test)

# ── Post-processing: ràng buộc nghiệp vụ ─────────────────────────────────────
pred_rev  = np.clip(pred_rev,  0, None)          # Revenue ≥ 0
pred_cogs = np.clip(pred_cogs, 0, None)          # COGS ≥ 0
pred_cogs = np.minimum(pred_cogs, pred_rev)      # COGS ≤ Revenue

print(f'Revenue  — min: {pred_rev.min():,.0f}  max: {pred_rev.max():,.0f}  mean: {pred_rev.mean():,.0f}')
print(f'COGS     — min: {pred_cogs.min():,.0f}  max: {pred_cogs.max():,.0f}  mean: {pred_cogs.mean():,.0f}')

---
## 8. Kiểm tra trực quan dự báo

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=False)

for ax, (title, train_col, pred_arr, color) in zip(
    axes,
    [
        ('Revenue: Train (180 ngày cuối) vs Forecast', 'Revenue', pred_rev,  '#1565C0'),
        ('COGS: Train (180 ngày cuối) vs Forecast',    'COGS',    pred_cogs, '#BF360C'),
    ]
):
    ax.plot(sales['Date'].iloc[-180:], sales[train_col].iloc[-180:],
            color='#546E7A', lw=1.2, alpha=0.8, label='Actual (train tail)')
    ax.plot(sample_sub['Date'], pred_arr,
            color=color, lw=1.5, label='Forecast')
    ax.axvline(sample_sub['Date'].iloc[0], color='red',
               linestyle='--', lw=1.2, label='Forecast start')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'
    ))
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)

plt.tight_layout()
save_path = OUT_DIR / 'forecast_plot.png'
fig.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Đã lưu → {save_path}')

---
## 9. Tạo file Submission

In [ ]:
submission = pd.DataFrame({
    'Date'   : df_test['Date'].dt.strftime('%Y-%m-%d'),
    'Revenue': np.round(pred_rev,  2),
    'COGS'   : np.round(pred_cogs, 2),
})

# ── Sanity checks ─────────────────────────────────────────────────────────────
assert len(submission) == len(sample_sub),           '❌ Số dòng không khớp sample_submission!'
assert (submission['Revenue'] >= 0).all(),           '❌ Revenue có giá trị âm!'
assert (submission['COGS']    >= 0).all(),           '❌ COGS có giá trị âm!'
assert (submission['COGS'] <= submission['Revenue']).all(), '❌ COGS > Revenue!'

out_path = OUT_DIR / 'submission.csv'
submission.to_csv(out_path, index=False)

print(f'✅ submission.csv đã tạo thành công!')
print(f'   Path : {out_path}')
print(f'   Shape: {submission.shape}')
print()
display(submission.head(5))
display(submission.tail(5))

---
## 10. Tóm tắt Pipeline & Reproducibility Checklist

### Pipeline

| Bước | Chi tiết |
|---|---|
| Feature Engineering | 20 features thuần từ Date — không dùng Revenue/COGS test (no leakage) |
| Target Transform | `log1p` → giảm ảnh hưởng outliers; inverse `expm1` khi predict |
| Cross-Validation | `TimeSeriesSplit(n_splits=5)` — đúng chiều thời gian |
| Ensemble | Trung bình dự báo của 5 fold models → giảm variance |
| Post-processing | `clip(0)` và `COGS ≤ Revenue` theo ràng buộc nghiệp vụ |
| Explainability | SHAP TreeExplainer — bar chart + beeswarm, sample cố định seed |

### ✅ Reproducibility Checklist

- [x] `SEED = 42` đặt tại đầu notebook, truyền vào `np.random.seed`, `LGBMRegressor(random_state=SEED)`, `df.sample(random_state=SEED)`
- [x] Tất cả hyperparameters được ghi rõ trong `LGB_PARAMS`
- [x] Không dùng dữ liệu ngoài bộ dataset cung cấp
- [x] Không dùng Revenue/COGS từ tập test làm feature
- [x] `TimeSeriesSplit` đảm bảo không data leakage theo thời gian
- [x] Toàn bộ mã nguồn trong GitHub repository kèm `README.md`